<a href="https://colab.research.google.com/github/kalingasajja/ML-learn/blob/main/Exp_6_lightgbm_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mlflow  lightgbm optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 677.0/677.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 8.1 MB/s eta 0:00:00


In [ ]:
import mlflow

mlflow.set_tracking_uri('https://56cf-2409-40f0-30a6-9a51-4084-42dc-b874-88bb.ngrok-free.app/')

In [ ]:
# Set or create an experiment
mlflow.set_experiment("Exp_4 TfIdf Trigram_1000_lightgbm")

2025/06/20 11:30:38 INFO mlflow.tracking.fluent: Experiment with name 'Exp_4 TfIdf Trigram_1000_lightgbm' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/581376276865275533', creation_time=1750419038092, experiment_id='581376276865275533', last_update_time=1750419038092, lifecycle_stage='active', name='Exp_4 TfIdf Trigram_1000_lightgbm', tags={}>

In [2]:
import mlflow
import mlflow.sklearn

from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,cross_val_predict,StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier



In [3]:
df = pd.read_csv("/content/reddit_preprocessing.csv")
df.shape

(36793, 2)

In [4]:
# Remap the class labels from [-1,0,1] to [2,0,1]
df['category'] = df['category'].map({-1:2,0:0,1:1})

# Remove null values
df = df.dropna(subset=['category'])
ngram_range = (1,3)
max_features = 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

X_train,X_test,y_train,y_test = train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42)
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
smote = SMOTE(random_state=42)
X_train,y_train = smote.fit_resample(X_train,y_train)



In [7]:
def log_mlflow(model_name,model,X_train,X_test,y_train,y_test,params,trail_number):

  if mlflow.active_run():
    mlflow.end_run()

  with mlflow.start_run():

    #Log model type
    mlflow.set_tag("mlflow.runName",f"{model_name}_SMOTE_TFIDF_Trigrams")
    mlflow.set_tag("experiment_type","algorithm_comparison")

    # Log algorithm name as parameter
    mlflow.log_param("algorithm_name",model_name)
    mlflow.log_params(params)

    # Train model
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    # Log accuracy
    # Log metrics for each class and accuracy
  accuracy = accuracy_score(y_test,y_pred)
  mlflow.log_metric("accuracy",accuracy)
  print(f"accuracy: {accuracy}")

  classification_rep = classification_report(y_test,y_pred,output_dict=True)

  for label , metrics in classification_rep.items():
    if isinstance(metrics,dict):
       # for precision , recall , f1-score, etc.,
       for metric,value in metrics.items():

         mlflow.log_metric(f"{label}_{metric}",value)

  # confusion Matrix plot
  conf_matrix = confusion_matrix(y_test,y_pred)
 # plt.figure(figsize=(8,6))
  sns.heatmap(conf_matrix,annot=True,fmt="d",cmap="Blues")
  #plt.title(f"Confusion Matrix: {model_name}")
  #plt.xlabel("Predicted")
  #plt.ylabel("Actual")

  # Save the plot to a temporary file name
  plot_filename = f"/confusion_matrix_tf_idf_{model_name}_model.png"
  plt.savefig(plot_filename)
  mlflow.log_artifact(plot_filename)

  # Log the Random Forest Model
  mlflow.sklearn.log_model(model,f"_tf_idf_{model_name}_model")
  return accuracy

In [5]:
# Optuna objective function for xgboost
def objective_function(trial):
  n_estimators = trial.suggest_int('n_estimators',50,300)
  learning_rate = trial.suggest_float('learning_rate',1e-4,1e-1,log=True)
  max_depth = trial.suggest_int('max_depth',3,15)
  num_leaves = trial.suggest_int('num_leaves',20,150)
  min_child_samples = trial.suggest_int("min_child_samples",10,100)
  colsample_bytree = trial.suggest_float("colsample_bytree",0.5,1.0)
  subsample = trial.suggest_float("subsample",0.5,1.0)
  reg_alpha = trial.suggest_float("reg_alpha",1e-4,10.0,log=True)  # L1 regularization
  reg_lambda = trial.suggest_float("reg_lambda",1e-4,10.0,log=True) # L2 regularization

  # Log trial parameters
  params = {
      'n_estmators' : n_estimators,
      'learning_rate' : learning_rate,
      'max_depth' : max_depth,
      'num_leaves' : num_leaves,
      'min_child_samples' : min_child_samples,
      'colsample_bytree' : colsample_bytree,
      'subsample' : subsample,
      'reg_alpha' : reg_alpha,
      'reg_lambda':reg_lambda
      }

  model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate,max_depth=max_depth,
                         num_leaves=num_leaves,
                         min_child_samples=min_child_samples,
                         colsample_bytree = colsample_bytree,
                         subsample=subsample,
                         reg_alpha=reg_alpha,
                         reg_lambda=reg_lambda,
                         random_state=42)
  accuracy= log_mlflow("LightGBM",model,X_train,X_test,y_train,y_test,params,trial.number)
  return accuracy


In [ ]:
# Run optuna for XGBoost and log the best model
def run_optuna_experiment():
  study = optuna.create_study(direction="maximize")
  study.optimize(objective_function,n_trials=10)

  # Get the best parameters and log only the best model
  best_params = study.best_params
  best_model = LGBMClassifier(n_estimators=best_params["n_estimators"],learning_rate=best_params['learning_rate'],max_depth=best_params['max_depth'],
                         num_leaves=best_params['num_leaves'],
                         min_child_samples=best_params['min_child_samples'],
                         colsample_bytree = best_params['colsample_bytree'],
                         subsample=best_params['subsample'],
                         reg_alpha=best_params['reg_alpha'],
                         reg_lambda=best_params['reg_lambda'],
                         random_state=42)
  log_mlflow("LightGBM",best_model,X_train,X_test,y_train,y_test,best_params,"Best")

  # Plot parameter importance
  optuna.visualization.plot_param_importances(study).show()

  # Plot optimization history
  optuna.visualization.plot_optimization_history(study).show()


run_optuna_experiment()


[I 2025-07-28 13:40:28,096] A new study created in memory with name: no-name-d3ddaf6a-0d1b-4686-9df7-85aae343048d


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.268323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 122646
[LightGBM] [Info] Number of data points in the train set: 38010, number of used features: 952
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] 

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


accuracy: 0.6431580377768719


2025/07/28 13:40:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/28 13:40:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
[I 2025-07-28 13:40:53,555] Trial 0 finished with value: 0.6431580377768719 and parameters: {'n_estimators': 137, 'learning_rate': 0.0015972219971032312, 'max_depth': 5, 'num_leaves': 140, 'min_child_samples': 93, 'colsample_bytree': 0.5920403786123072, 'subsample': 0.6940224618627867, 'reg_alpha': 0.012431718433853036, 'reg_lambda': 0.00010917416984080107}. Best is trial 0 with value: 0.6431580377768719.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.260730 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 122739
[LightGBM] [Info] Number of data points in the train set: 38010, number of used features: 955
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


accuracy: 0.7082484033156679


2025/07/28 13:43:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/28 13:43:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
[I 2025-07-28 13:43:47,302] Trial 7 finished with value: 0.7082484033156679 and parameters: {'n_estimators': 132, 'learning_rate': 0.0056570218035516175, 'max_depth': 12, 'num_leaves': 28, 'min_child_samples': 77, 'colsample_bytree': 0.6897954996442297, 'subsample': 0.958912853570576, 'reg_alpha': 0.8879912176526714, 'reg_lambda': 0.0059019260627353656}. Best is trial 3 with value: 0.7816279385786112.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.265368 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 123016
[LightGBM] [Info] Number of data points in the train set: 38010, number of used features: 968
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 